In [5]:
# necessary imports

import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt
import statsmodels.api as sm

sns.set_style("whitegrid")

In [6]:
# read in csv

merged = pd.read_csv("../Final Project/merged_arrests_acs_2023.csv")
merged.head()

,ZCTA,total_arrests,felonies,misdemeanors,violations,AMERICAN INDIAN/ALASKAN NATIVE,ASIAN / PACIFIC ISLANDER,BLACK,BLACK HISPANIC,UNKNOWN,...,pct_native,pct_asian,pct_hispanic,arrest_rate_per_1000,HISPANIC_ARRESTS,black_arrest_rate_per_1000,white_arrest_rate_per_1000,asian_pacific_islander_arrest_rate_per_1000,american_indian/alaskan_native_arrest_rate_per_1000,hispanic_arrest_rate_per_1000
0,83,172,75,97,0,4,3,65,20,6,...,NaN,NaN,NaN,NaN,63,NaN,NaN,NaN,NaN,NaN
1,10001,4041,1660,2291,8,14,116,1798,371,106,...,0.000000,0.179786,0.190756,138.966264,1515,615.121451,34.596723,22.188217,NaN,273.120606
2,10002,2107,947,1137,8,1,166,924,209,24,...,0.000914,0.365441,0.246911,27.901002,780,140.106141,10.655408,6.015147,14.492754,41.832028
3,10003,1742,856,877,8,4,60,822,156,32,...,0.001263,0.178913,0.098189,32.364143,559,348.452734,7.685615,6.230530,58.823529,105.771050
4,10004,61,17,44,0,0,5,31,2,2,...,0.000000,0.238194,0.048774,15.741935,13,116.104869,4.187605,5.417118,NaN,68.783069


In [7]:
# general info and summary stats

print(merged.info())
print(merged.describe(include="all").transpose().head(20))

# count missing by column

missing = merged.isna().sum().sort_values(ascending=False)
missing[missing > 0]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 32 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   ZCTA                                                 190 non-null    int64  
 1   total_arrests                                        190 non-null    int64  
 2   felonies                                             190 non-null    int64  
 3   misdemeanors                                         190 non-null    int64  
 4   violations                                           190 non-null    int64  
 5   AMERICAN INDIAN/ALASKAN NATIVE                       190 non-null    int64  
 6   ASIAN / PACIFIC ISLANDER                             190 non-null    int64  
 7   BLACK                                                190 non-null    int64  
 8   BLACK HISPANIC                                       190 non-null    i

american_indian/alaskan_native_arrest_rate_per_1000    50
black_arrest_rate_per_1000                             11
hispanic_arrest_rate_per_1000                           9
asian_pacific_islander_arrest_rate_per_1000             9
white_arrest_rate_per_1000                              9
pct_hispanic                                            9
pct_asian                                               9
pct_native                                              9
pct_black                                               9
pct_white                                               9
native_count                                            4
arrest_rate_per_1000                                    4
below_poverty                                           4
median_income                                           4
hispanic_count                                          4
asian_count                                             4
black_count                                             4
white_count   

In [8]:
# drop rows where total_pop is missing or 0

df = merged.dropna(subset=["total_pop"]).copy()
df = df[df["total_pop"] > 0]

In [9]:
# ensure cols to numeric 

numeric_cols = [
    "total_arrests", "felonies", "misdemeanors", "violations",
    "total_pop", "white_count", "black_count", "native_count",
    "asian_count", "hispanic_count", "median_income", "below_poverty",
    "pct_white", "pct_black", "pct_native", "pct_asian", "pct_hispanic",
    "arrest_rate_per_1000"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [10]:
# drop insane values

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["arrest_rate_per_1000"])

In [11]:
# remove extreme outliers

upper_limit = df["arrest_rate_per_1000"].mean() + 3 * df["arrest_rate_per_1000"].std()
df = df[df["arrest_rate_per_1000"] < upper_limit]

In [12]:
# check key cols
 
key_cols = ["median_income", "pct_black", "pct_hispanic", "pct_asian", "pct_white"]
print(df[key_cols].isna().sum())

print(f"Final ZIPs retained: {df.shape[0]}")
df.describe().T.loc[["total_arrests", "total_pop", "arrest_rate_per_1000"]]

median_income    0
pct_black        0
pct_hispanic     0
pct_asian        0
pct_white        0
dtype: int64
Final ZIPs retained: 178


,count,mean,std,min,25%,50%,75%,max
total_arrests,178.0,1098.095506,951.950627,1.000000,329.250000,831.000000,1684.750000,5360.000000
total_pop,178.0,47699.747191,26151.578964,2195.000000,28003.500000,42456.500000,68430.750000,107060.000000
arrest_rate_per_1000,178.0,22.514149,17.818314,0.242542,9.727657,17.510165,31.969244,90.590112


In [13]:
df = df[df["total_pop"] > 20000].copy()
print(f"ZIPs kept: {df.shape[0]}")

ZIPs kept: 151


In [14]:
import pandas as pd, json, os
import plotly.express as px

CSV = "../Final Project/merged_arrests_acs_2023.csv"
GEOJSON = "../Final Project/nyc-zip-code-tabulation-areas.geojson"

df = pd.read_csv(CSV)

zip_cols = [c for c in df.columns if any(s in c.lower() for s in ['zip','zcta','geoid'])]
df['zip5'] = None
for c in zip_cols:
    s = df[c].astype(str).str.extract(r'(\d{5})', expand=False)
    if s.notna().mean() > 0.7:
        df['zip5'] = s.str.zfill(5)
        break
if df['zip5'].isna().all():
    
    df['zip5'] = df.index.astype(str).str.extract(r'(\d{5})', expand=False).str.zfill(5)

VALUE_COL = 'arrest_rate_per_1000'

with open(GEOJSON, "r") as f:
    gj = json.load(f)

FEATURE_ID_KEY = "properties.postalCode"  

#interactive choropleth
fig = px.choropleth(
    df,
    geojson=gj,
    locations="zip5",
    color=VALUE_COL,
    featureidkey=FEATURE_ID_KEY,
    color_continuous_scale="Viridis",
    hover_data={c: True for c in df.columns if c != "zip5"},
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    title=f"NYC ZIP Choropleth — {VALUE_COL}",
    margin={"r":0,"t":50,"l":0,"b":0},
    coloraxis_colorbar_title=VALUE_COL
)

OUT = "../Final Project/nyc_choropleth_plotly.html"
fig.write_html(OUT, include_plotlyjs="cdn")
print(f"Saved: {OUT}")


Saved: ../Final Project/nyc_choropleth_plotly.html


In [15]:
df[['zip5','arrest_rate_per_1000']].sort_values('arrest_rate_per_1000', ascending=False).head(10)


,zip5,arrest_rate_per_1000
18,10020,inf
152,11371,inf
184,11451,inf
46,10169,inf
44,10111,inf
16,10018,145.367412
1,10001,138.966264
178,11430,133.928571
12,10013,90.590112
7,10007,88.695206
